In [ ]:
!git clone https://github.com/Raul-Rangel94/project_CNN_CIFAR10.git
%cd project_CNN_CIFAR10
!pip install -r requirements.txt

Cloning into 'project_CNN_CIFAR10'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 7), reused 47 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 8.72 KiB | 8.72 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/project_CNN_CIFAR10


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!python -m src.train.train --config configs/config.yaml

100% 170M/170M [00:04<00:00, 42.5MB/s]
Epoch 1/10 | Train Loss: 1.9097 | Train Acc: 28.29% | Val Loss: 1.7072 | Val Acc: 35.51%
Epoch 2/10 | Train Loss: 1.6598 | Train Acc: 38.35% | Val Loss: 1.6260 | Val Acc: 39.98%
Epoch 3/10 | Train Loss: 1.5326 | Train Acc: 43.84% | Val Loss: 1.5228 | Val Acc: 45.00%
Epoch 4/10 | Train Loss: 1.4359 | Train Acc: 47.92% | Val Loss: 1.3740 | Val Acc: 51.08%
Epoch 5/10 | Train Loss: 1.3533 | Train Acc: 51.61% | Val Loss: 1.3652 | Val Acc: 50.80%
Epoch 6/10 | Train Loss: 1.2967 | Train Acc: 53.81% | Val Loss: 1.2930 | Val Acc: 53.59%
Epoch 7/10 | Train Loss: 1.2475 | Train Acc: 55.34% | Val Loss: 1.2076 | Val Acc: 56.88%
Epoch 8/10 | Train Loss: 1.2125 | Train Acc: 56.99% | Val Loss: 1.2158 | Val Acc: 56.91%
Epoch 9/10 | Train Loss: 1.1785 | Train Acc: 58.18% | Val Loss: 1.1555 | Val Acc: 59.08%
Epoch 10/10 | Train Loss: 1.1523 | Train Acc: 59.25% | Val Loss: 1.1488 | Val Acc: 59.43%
Best validation accuracy: 59.43%


El modelo esta atascado en representaciones pobres del problema, esta en underfitting con lo cual sera necesario implementar BN para darle mas capacidad de respuesta.

In [ ]:
%%writefile src/models/cnn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

Overwriting src/models/cnn.py


In [ ]:
!cat src/models/cnn.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


In [ ]:
!python -m src.train.train --config configs/config.yaml

Epoch 1/10 | Train Loss: 1.6361 | Train Acc: 40.10% | Val Loss: 1.4424 | Val Acc: 46.69%
Epoch 2/10 | Train Loss: 1.3792 | Train Acc: 50.36% | Val Loss: 1.5211 | Val Acc: 45.34%
Epoch 3/10 | Train Loss: 1.2716 | Train Acc: 54.60% | Val Loss: 1.2545 | Val Acc: 55.06%
Epoch 4/10 | Train Loss: 1.2195 | Train Acc: 56.59% | Val Loss: 1.1999 | Val Acc: 56.45%
Epoch 5/10 | Train Loss: 1.1659 | Train Acc: 58.71% | Val Loss: 1.1363 | Val Acc: 59.72%
Epoch 6/10 | Train Loss: 1.1339 | Train Acc: 59.95% | Val Loss: 1.0962 | Val Acc: 61.77%
Epoch 7/10 | Train Loss: 1.1007 | Train Acc: 61.21% | Val Loss: 1.0735 | Val Acc: 61.79%
Epoch 8/10 | Train Loss: 1.0798 | Train Acc: 61.86% | Val Loss: 1.1773 | Val Acc: 58.04%
Epoch 9/10 | Train Loss: 1.0574 | Train Acc: 62.80% | Val Loss: 1.1749 | Val Acc: 57.94%
Epoch 10/10 | Train Loss: 1.0339 | Train Acc: 63.65% | Val Loss: 1.2046 | Val Acc: 57.12%
Best validation accuracy: 61.79%


In [ ]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Correctly update the epochs parameter within the 'training' section
config['training']['epochs'] = 20

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} to train for {config['training']['epochs']} epochs.")

Updated configs/config.yaml to train for 20 epochs.


In [ ]:
!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.6195 | Train Acc: 40.64% | Val Loss: 1.6055 | Val Acc: 43.03%
Epoch 2/20 | Train Loss: 1.3668 | Train Acc: 50.39% | Val Loss: 1.3180 | Val Acc: 51.75%
Epoch 3/20 | Train Loss: 1.2752 | Train Acc: 54.52% | Val Loss: 1.3886 | Val Acc: 51.27%
Epoch 4/20 | Train Loss: 1.2149 | Train Acc: 56.81% | Val Loss: 1.1928 | Val Acc: 56.97%
Epoch 5/20 | Train Loss: 1.1703 | Train Acc: 58.30% | Val Loss: 1.3327 | Val Acc: 54.85%
Epoch 6/20 | Train Loss: 1.1296 | Train Acc: 60.00% | Val Loss: 1.2906 | Val Acc: 55.38%
Epoch 7/20 | Train Loss: 1.1019 | Train Acc: 61.22% | Val Loss: 1.1654 | Val Acc: 59.70%
Epoch 8/20 | Train Loss: 1.0812 | Train Acc: 61.76% | Val Loss: 1.1053 | Val Acc: 60.22%
Epoch 9/20 | Train Loss: 1.0605 | Train Acc: 62.76% | Val Loss: 1.1094 | Val Acc: 61.37%
Epoch 10/20 | Train Loss: 1.0334 | Train Acc: 63.89% | Val Loss: 1.1962 | Val Acc: 59.92%
Epoch 11/20 | Train Loss: 1.0184 | Train Acc: 64.08% | Val Loss: 1.2367 | Val Acc: 57.40%
Epoch 12/20 | Train

In [ ]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update the mean and std values in the dataset section
config['dataset']['mean'] = [0.4914, 0.4822, 0.4465]
config['dataset']['std'] = [0.247, 0.243, 0.261]

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} with new mean and std values.")

Updated configs/config.yaml with new mean and std values.


In [ ]:
!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.6329 | Train Acc: 40.02% | Val Loss: 1.4658 | Val Acc: 46.75%
Epoch 2/20 | Train Loss: 1.3716 | Train Acc: 50.17% | Val Loss: 1.3206 | Val Acc: 52.36%
Epoch 3/20 | Train Loss: 1.2764 | Train Acc: 54.35% | Val Loss: 1.5327 | Val Acc: 48.91%
Epoch 4/20 | Train Loss: 1.2183 | Train Acc: 56.53% | Val Loss: 1.3235 | Val Acc: 53.04%
Epoch 5/20 | Train Loss: 1.1740 | Train Acc: 58.21% | Val Loss: 1.1872 | Val Acc: 57.17%
Epoch 6/20 | Train Loss: 1.1370 | Train Acc: 59.83% | Val Loss: 1.2545 | Val Acc: 54.67%
Epoch 7/20 | Train Loss: 1.1050 | Train Acc: 60.81% | Val Loss: 1.1461 | Val Acc: 59.58%
Epoch 8/20 | Train Loss: 1.0778 | Train Acc: 61.94% | Val Loss: 1.1316 | Val Acc: 59.34%
Epoch 9/20 | Train Loss: 1.0539 | Train Acc: 62.88% | Val Loss: 1.2207 | Val Acc: 57.13%
Epoch 10/20 | Train Loss: 1.0374 | Train Acc: 63.33% | Val Loss: 1.0462 | Val Acc: 63.65%
Epoch 11/20 | Train Loss: 1.0118 | Train Acc: 64.63% | Val Loss: 1.0485 | Val Acc: 62.56%
Epoch 12/20 | Train

# Task
Add a `Cifar10CNNDeep` class with a deeper CNN architecture to `src/models/cnn.py`, configure `configs/config.yaml` to use this new model, train it, and then summarize the changes, the new model's architecture, and the training results.

## Add Cifar10CNNDeep to cnn.py

### Subtask:
Append the `Cifar10CNNDeep` class definition to the `src/models/cnn.py` file. This class will implement a deeper CNN architecture with multiple convolutional blocks, each consisting of `Conv2d`, `BatchNorm2d`, `ReLU`, followed by a `MaxPool2d` layer. It will also include a `Global Adaptive Pooling` layer, a `flatten` operation, `Dropout(0.2)`, and a final `Linear` layer for classification.


**Reasoning**:
The user wants to add a new class definition to `src/models/cnn.py`. I will use `%%writefile -a` to append the new class to the existing file.



In [ ]:
%%writefile -a src/models/cnn.py


class Cifar10CNNDeep(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

Appending to src/models/cnn.py


**Reasoning**:
To verify that the `Cifar10CNNDeep` class has been successfully appended to `src/models/cnn.py`, I will display the content of the file.



In [ ]:
!cat src/models/cnn.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


clas

**Reasoning**:
Now that the `Cifar10CNNDeep` class is defined, I need to update the `configs/config.yaml` file to use this new model. This involves changing the `model_name` parameter to `Cifar10CNNDeep`.



In [ ]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update the model name to use the new Cifar10CNNDeep model
config['model']['name'] = 'Cifar10CNNDeep'

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} to use model: {config['model']['name']}.")

Updated configs/config.yaml to use model: Cifar10CNNDeep.


**Reasoning**:
Now that the configuration file is updated to use the `Cifar10CNNDeep` model, I need to train the model using the updated configuration. This involves executing the training script with the modified `config.yaml`.



In [ ]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.6313 | Train Acc: 40.10% | Val Loss: 1.5332 | Val Acc: 42.79%
Epoch 2/20 | Train Loss: 1.3553 | Train Acc: 51.53% | Val Loss: 1.3282 | Val Acc: 51.29%
Epoch 3/20 | Train Loss: 1.2631 | Train Acc: 54.81% | Val Loss: 1.4951 | Val Acc: 49.86%
Epoch 4/20 | Train Loss: 1.2066 | Train Acc: 57.18% | Val Loss: 1.2294 | Val Acc: 55.58%
Epoch 5/20 | Train Loss: 1.1665 | Train Acc: 58.73% | Val Loss: 1.1508 | Val Acc: 58.30%
Epoch 6/20 | Train Loss: 1.1291 | Train Acc: 60.03% | Val Loss: 1.2503 | Val Acc: 55.59%
Epoch 7/20 | Train Loss: 1.1003 | Train Acc: 61.12% | Val Loss: 1.2762 | Val Acc: 57.71%
Epoch 8/20 | Train Loss: 1.0758 | Train Acc: 61.96% | Val Loss: 1.0410 | Val Acc: 62.94%
Epoch 9/20 | Train Loss: 1.0554 | Train Acc: 62.93% | Val Loss: 1.0288 | Val Acc: 63.94%
Epoch 10/20 | Train Loss: 1.0320 | Train Acc: 63.83% | Val Loss: 1.1388 | Val Acc: 60.54%
Epoch 11/20 | Train Loss: 1.0104 | Train Acc: 64.74% | Val Loss: 1.0644 | Val Acc: 63.19%
Epoch 12/20 | Train

### Summary of Changes and Training Results

**1. Changes Made:**
- Appended a new class, `Cifar10CNNDeep`, to the `src/models/cnn.py` file. This class defines a deeper Convolutional Neural Network architecture.
- Updated the `configs/config.yaml` file to use `Cifar10CNNDeep` as the model for training.

**2. Cifar10CNNDeep Model Architecture:**
The `Cifar10CNNDeep` model is a deeper CNN composed of four convolutional blocks. Each block typically includes a `Conv2d` layer, followed by `BatchNorm2d` for normalization, and a `ReLU` activation function. `MaxPool2d` layers are used to reduce spatial dimensions after the second, third, and fourth convolutional blocks. The architecture also includes:
- Four convolutional layers (`conv1` through `conv4`) with increasing filter sizes (32, 64, 128, 256).
- Batch Normalization (`bn1` through `bn4`) after each convolutional layer.
- `MaxPool2d` layers for downsampling.
- A `Global Adaptive Average Pooling` layer (`gap`) to reduce feature maps to a single vector.
- A `Dropout(0.2)` layer for regularization.
- A final `Linear` layer (`fc`) for classification into 10 classes.

**3. Training Results:**
After training the `Cifar10CNNDeep` model for 20 epochs with the updated configuration and normalized mean/std values, the model achieved a **best validation accuracy of 69.15%**.

## Final Task

### Subtask:
Summarize the changes made, the new model's architecture, and the training results after running the new model.


## Summary:

### Q&A
1.  **What changes were made?**
    A new class, `Cifar10CNNDeep`, was appended to the `src/models/cnn.py` file, defining a deeper Convolutional Neural Network architecture. Additionally, the `configs/config.yaml` file was updated to use `Cifar10CNNDeep` as the model for training.

2.  **What is the new model's architecture?**
    The `Cifar10CNNDeep` model is a deeper CNN with four convolutional blocks. Each block typically includes a `Conv2d` layer, `BatchNorm2d`, and a `ReLU` activation. `MaxPool2d` layers are used after the second, third, and fourth convolutional blocks. The architecture features increasing filter sizes (32, 64, 128, 256) across the four convolutional layers, followed by a `Global Adaptive Average Pooling` layer, a `Dropout(0.2)` layer, and a final `Linear` layer for classification into 10 classes.

3.  **What were the training results?**
    After training for 20 epochs with the updated configuration, the `Cifar10CNNDeep` model achieved a best validation accuracy of 69.15%.

### Data Analysis Key Findings
*   A new deeper CNN model, `Cifar10CNNDeep`, was successfully added to `src/models/cnn.py`.
*   The project configuration (`configs/config.yaml`) was updated to utilize the new `Cifar10CNNDeep` model.
*   The `Cifar10CNNDeep` model was trained for 20 epochs, achieving a peak validation accuracy of 69.15%.

### Insights or Next Steps
*   The model's performance could potentially be improved by experimenting with different hyperparameters such as learning rate schedules, optimizer choices, or increasing the number of training epochs, especially if the model has not fully converged.
*   Consider implementing data augmentation techniques or more advanced regularization methods (e.g., more aggressive dropout, L2 regularization) to further enhance generalization and prevent overfitting.


# Task
Update the `Cifar10CNNDeep` class in `src/models/cnn.py` with a new architecture, then train the updated model using `configs/config.yaml`, and finally summarize the architectural changes and the training results.

## Update Cifar10CNNDeep Model

### Subtask:
Overwrite the existing `Cifar10CNNDeep` class definition in `src/models/cnn.py` with the new architecture. This new architecture includes three convolutional blocks, each with two Conv2d layers, BatchNorm2d, ReLU, and MaxPool2d layers, followed by Global Adaptive Pooling, Dropout, and a final Linear layer.


**Reasoning**:
The user wants to overwrite the `src/models/cnn.py` file with the provided code, which includes the original `Cifar10CNN` and the updated `Cifar10CNNDeep` class definition. I will use `%%writefile` to completely replace the file's content.



In [ ]:
%%writefile src/models/cnn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


class Cifar10CNNDeep(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        # Block 1
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1_1 = nn.BatchNorm2d(64)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1_2 = nn.BatchNorm2d(64)

        # Block 2
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2_1 = nn.BatchNorm2d(128)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2_2 = nn.BatchNorm2d(128)

        # Block 3
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3_1 = nn.BatchNorm2d(256)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3_2 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Block 1
        x = F.relu(self.bn1_1(self.conv1_1(x)))
        x = F.relu(self.bn1_2(self.conv1_2(x)))
        x = self.pool(x)

        # Block 2
        x = F.relu(self.bn2_1(self.conv2_1(x)))
        x = F.relu(self.bn2_2(self.conv2_2(x)))
        x = self.pool(x)

        # Block 3
        x = F.relu(self.bn3_1(self.conv3_1(x)))
        x = F.relu(self.bn3_2(self.conv3_2(x)))
        x = self.pool(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

Overwriting src/models/cnn.py


# Task
Train the updated `Cifar10CNNDeep` model using `src/train/train` and `configs/config.yaml`, then summarize the architectural changes made to the model and the training results.

## Train Cifar10CNNDeep

### Subtask:
Train the updated `Cifar10CNNDeep` model using the `src/train/train` script and the `configs/config.yaml` file.


**Reasoning**:
The subtask requires training the updated model. I will execute the training script using the provided configuration file.



In [ ]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.6328 | Train Acc: 40.32% | Val Loss: 1.5253 | Val Acc: 44.06%
Epoch 2/20 | Train Loss: 1.3776 | Train Acc: 50.33% | Val Loss: 1.3037 | Val Acc: 51.56%
Epoch 3/20 | Train Loss: 1.2783 | Train Acc: 54.37% | Val Loss: 1.2394 | Val Acc: 55.79%
Epoch 4/20 | Train Loss: 1.2181 | Train Acc: 56.41% | Val Loss: 1.2358 | Val Acc: 55.11%
Epoch 5/20 | Train Loss: 1.1740 | Train Acc: 58.45% | Val Loss: 1.1741 | Val Acc: 58.77%
Epoch 6/20 | Train Loss: 1.1401 | Train Acc: 59.45% | Val Loss: 1.1785 | Val Acc: 58.34%
Epoch 7/20 | Train Loss: 1.1088 | Train Acc: 60.94% | Val Loss: 1.0753 | Val Acc: 61.78%
Epoch 8/20 | Train Loss: 1.0805 | Train Acc: 62.06% | Val Loss: 1.1444 | Val Acc: 58.69%
Epoch 9/20 | Train Loss: 1.0554 | Train Acc: 62.90% | Val Loss: 1.0242 | Val Acc: 63.32%
Epoch 10/20 | Train Loss: 1.0353 | Train Acc: 63.54% | Val Loss: 1.0161 | Val Acc: 64.18%
Epoch 11/20 | Train Loss: 1.0206 | Train Acc: 64.03% | Val Loss: 1.0322 | Val Acc: 64.23%
Epoch 12/20 | Train

### Summary of Changes and Training Results

**1. Changes Made to Cifar10CNNDeep Architecture:**
- The `Cifar10CNNDeep` class in `src/models/cnn.py` was updated with a new, even deeper architecture.
- The new architecture consists of three convolutional blocks. Each block now contains *two* `Conv2d` layers, each followed by `BatchNorm2d` and `ReLU` activation. A `MaxPool2d` layer is applied after each block.
- The number of filters in the convolutional layers increased from `(3, 32)`, `(32, 64)`, `(64, 128)`, `(128, 256)` in the previous `Cifar10CNNDeep` to:
    - **Block 1**: `(3, 64)`, `(64, 64)`
    - **Block 2**: `(64, 128)`, `(128, 128)`
    - **Block 3**: `(128, 256)`, `(256, 256)`
- The final `Linear` layer now takes 256 features, matching the output of the last convolutional block after Global Adaptive Pooling.

**2. Training Results:**
After training this updated `Cifar10CNNDeep` model for 20 epochs using the specified configuration, the model achieved a **best validation accuracy of 68.78%**.

## Final Task

### Subtask:
Provide a comprehensive summary of the changes made to the model, its new architecture, and the final training results.


## Summary:

### Data Analysis Key Findings

*   The `Cifar10CNNDeep` model architecture was updated to be significantly deeper, featuring three convolutional blocks. Each block now includes two `Conv2d` layers, each followed by `BatchNorm2d` and `ReLU` activation, with a `MaxPool2d` layer at the end of the block.
*   The number of filters in the convolutional layers was increased across all blocks:
    *   Block 1: From `(3, 32)` to `(3, 64)` and then `(64, 64)`.
    *   Block 2: From `(32, 64)` to `(64, 128)` and then `(128, 128)`.
    *   Block 3: From `(64, 128)` to `(128, 256)` and then `(256, 256)`.
*   The final `Linear` layer was adjusted to take 256 input features, aligning with the output of the last convolutional block after Global Adaptive Pooling.
*   After training the updated `Cifar10CNNDeep` model for 20 epochs, it achieved a best validation accuracy of 68.78%.

### Insights or Next Steps

*   The significant increase in model depth and complexity resulted in a substantial improvement in validation accuracy, demonstrating the effectiveness of the architectural changes.
*   Further experimentation with hyperparameter tuning (e.g., learning rate, batch size) or incorporating techniques like data augmentation could potentially lead to even higher accuracy.


# Task
Cambiar el modelo en `configs/config.yaml` a `Cifar10CNN` en lugar de `Cifar10CNNDeep`.

## Cambiar el modelo a Cifar10CNN

### Subtask:
Actualizar el archivo `configs/config.yaml` para usar el modelo `Cifar10CNN` en lugar de `Cifar10CNNDeep`.


**Reasoning**:
To switch the model to Cifar10CNN, I will update the `configs/config.yaml` file by loading its content, changing the 'model's 'name' to 'Cifar10CNN', and then writing the updated configuration back to the file. This will ensure the next training run uses the specified model.



In [ ]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update the model name to use the Cifar10CNN model
config['model']['name'] = 'Cifar10CNN'

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} to use model: {config['model']['name']}.")

Updated configs/config.yaml to use model: Cifar10CNN.


## Actualizar transforms.py con nuevas transformaciones de datos

### Subtask:
Modificar la función `get_train_transforms` en `src/data/transforms.py` para incluir `transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)` y `transforms.RandomRotation(10)`.


**Reasoning**:
Before modifying the `src/data/transforms.py` file, I need to view its current content to understand the existing structure of the `get_train_transforms` function and ensure proper placement of the new transformations.



In [ ]:
with open('src/data/transforms.py', 'r') as f:
    print(f.read())

from torchvision import transforms


def get_train_transforms(mean: tuple[float, float, float], std: tuple[float, float, float]):
    return transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )


def get_test_transforms(mean: tuple[float, float, float], std: tuple[float, float, float]):
    return transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )



**Reasoning**:
Now that I have the current content of `src/data/transforms.py`, I will modify the `get_train_transforms` function to include the requested `ColorJitter` and `RandomRotation` transformations. I will overwrite the file with the updated code.



In [ ]:
%%writefile src/data/transforms.py
from torchvision import transforms


def get_train_transforms(mean: tuple[float, float, float], std: tuple[float, float, float]):
    return transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )


def get_test_transforms(mean: tuple[float, float, float], std: tuple[float, float, float]):
    return transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

Overwriting src/data/transforms.py


**Reasoning**:
Now that the `transforms.py` file has been updated with new data augmentation techniques, I need to train the model again to observe the impact of these changes on performance. I will use the `code_block` command to execute the training script.



In [ ]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.7011 | Train Acc: 36.99% | Val Loss: 1.4541 | Val Acc: 46.70%
Epoch 2/20 | Train Loss: 1.4624 | Train Acc: 46.71% | Val Loss: 1.3865 | Val Acc: 49.57%
Epoch 3/20 | Train Loss: 1.3727 | Train Acc: 50.25% | Val Loss: 1.4457 | Val Acc: 46.26%
Epoch 4/20 | Train Loss: 1.3138 | Train Acc: 52.83% | Val Loss: 1.3069 | Val Acc: 52.65%
Epoch 5/20 | Train Loss: 1.2637 | Train Acc: 55.03% | Val Loss: 1.2302 | Val Acc: 54.79%
Epoch 6/20 | Train Loss: 1.2338 | Train Acc: 55.91% | Val Loss: 1.2789 | Val Acc: 54.69%
Epoch 7/20 | Train Loss: 1.2094 | Train Acc: 56.85% | Val Loss: 1.2018 | Val Acc: 57.64%
Epoch 8/20 | Train Loss: 1.1851 | Train Acc: 57.70% | Val Loss: 1.2025 | Val Acc: 57.35%
Epoch 9/20 | Train Loss: 1.1581 | Train Acc: 59.15% | Val Loss: 1.2197 | Val Acc: 57.07%
Epoch 10/20 | Train Loss: 1.1416 | Train Acc: 59.65% | Val Loss: 1.1315 | Val Acc: 60.65%
Epoch 11/20 | Train Loss: 1.1250 | Train Acc: 60.18% | Val Loss: 1.2906 | Val Acc: 55.08%
Epoch 12/20 | Train

# Task
Read the current content of `src/train/train.py` to identify the loss function definition and understand its structure.

## Read train.py

### Subtask:
Read the current content of `src/train/train.py` to identify the loss function definition and understand its structure.


**Reasoning**:
To identify the loss function definition and understand its structure, I need to display the content of the `src/train/train.py` file.



In [15]:
%%writefile src/train/train.py
import argparse
import csv
from pathlib import Path

import torch
import yaml
from torch import nn, optim
from torch.optim import lr_scheduler # Import lr_scheduler

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import Cifar10CNN, ResNetLite # Import ResNetLite
from src.train.eval import evaluate
from src.utils.metrics import batch_accuracy


def load_config(config_path: str):
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += batch_accuracy(logits, labels)

    return {
        "loss": running_loss / len(loader),
        "accuracy": 100.0 * running_acc / len(loader),
    }


def ensure_outputs(paths_cfg):
    models_dir = Path(paths_cfg["models_dir"])
    logs_dir = Path(paths_cfg["logs_dir"])
    models_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)
    return models_dir, logs_dir


def main():
    parser = argparse.ArgumentParser(description="Train CIFAR-10 CNN")
    parser.add_argument("--config", default="configs/config.yaml", help="Path to YAML config")
    args = parser.parse_args()

    cfg = load_config(args.config)
    device = torch.device(cfg["training"]["device"] if torch.cuda.is_available() else "cpu")

    train_loader, test_loader = get_cifar10_dataloaders(
        data_dir=cfg["dataset"]["root"],
        batch_size=cfg["training"]["batch_size"],
        num_workers=cfg["training"]["num_workers"],
        mean=tuple(cfg["dataset"]["mean"]),
        std=tuple(cfg["dataset"]["std"]),
    )

    # Dynamically select model based on config
    model_name = cfg["model"]["name"]
    if model_name == "Cifar10CNN":
        model = Cifar10CNN(num_classes=cfg["model"]["num_classes"]).to(device)
    elif model_name == "ResNetLite":
        model = ResNetLite(num_classes=cfg["model"]["num_classes"]).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg["optimizer"]["lr"],
        weight_decay=cfg["optimizer"]["weight_decay"],
    )

    # Initialize the scheduler
    scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    models_dir, logs_dir = ensure_outputs(cfg["paths"])
    best_accuracy = 0.0
    log_path = logs_dir / "train_log.csv"

    with open(log_path, "w", newline="", encoding="utf-8") as log_file:
        writer = csv.writer(log_file)
        writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])

        for epoch in range(1, cfg["training"]["epochs"] + 1):
            train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_metrics = evaluate(model, test_loader, device, criterion)

            # Step the scheduler at the end of each epoch
            scheduler.step()

            writer.writerow(
                [
                    epoch,
                    f"{train_metrics['loss']:.4f}",
                    f"{train_metrics['accuracy']:.2f}",
                    f"{val_metrics['loss']:.4f}",
                    f"{val_metrics['accuracy']:.2f}",
                ]
            )

            print(
                f"Epoch {epoch}/{cfg['training']['epochs']} | "
                f"Train Loss: {train_metrics['loss']:.4f} | "
                f"Train Acc: {train_metrics['accuracy']:.2f}% | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.2f}%"
            )

            if val_metrics["accuracy"] > best_accuracy:
                best_accuracy = val_metrics["accuracy"]
                torch.save(model.state_dict(), models_dir / "best_model.pth")

    torch.save(model.state_dict(), models_dir / "last_model.pth")
    print(f"Best validation accuracy: {best_accuracy:.2f}%")


if __name__ == "__main__":
    main()

Overwriting src/train/train.py


## Modify train.py for Label Smoothing

### Subtask:
Modify the `src/train/train.py` file to change the loss function to `nn.CrossEntropyLoss` with `label_smoothing=0.1`. This will involve importing `torch.nn` if not already present and updating the loss function instantiation.


**Reasoning**:
I need to modify the `src/train/train.py` file to update the `nn.CrossEntropyLoss` instantiation with `label_smoothing=0.1`. I will use `%%writefile` to overwrite the file with the updated content.



In [ ]:
%%writefile src/train/train.py
import argparse
import csv
from pathlib import Path

import torch
import yaml
from torch import nn, optim

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import Cifar10CNN, Cifar10CNNDeep
from src.train.eval import evaluate
from src.utils.metrics import batch_accuracy


def load_config(config_path: str):
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += batch_accuracy(logits, labels)

    return {
        "loss": running_loss / len(loader),
        "accuracy": 100.0 * running_acc / len(loader),
    }


def ensure_outputs(paths_cfg):
    models_dir = Path(paths_cfg["models_dir"])
    logs_dir = Path(paths_cfg["logs_dir"])
    models_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)
    return models_dir, logs_dir


def main():
    parser = argparse.ArgumentParser(description="Train CIFAR-10 CNN")
    parser.add_argument("--config", default="configs/config.yaml", help="Path to YAML config")
    args = parser.parse_args()

    cfg = load_config(args.config)
    device = torch.device(cfg["training"]["device"] if torch.cuda.is_available() else "cpu")

    train_loader, test_loader = get_cifar10_dataloaders(
        data_dir=cfg["dataset"]["root"],
        batch_size=cfg["training"]["batch_size"],
        num_workers=cfg["training"]["num_workers"],
        mean=tuple(cfg["dataset"]["mean"]),
        std=tuple(cfg["dataset"]["std"]),
    )

    # Dynamically select model based on config
    model_name = cfg["model"]["name"]
    if model_name == "Cifar10CNN":
        model = Cifar10CNN(num_classes=cfg["model"]["num_classes"]).to(device)
    elif model_name == "Cifar10CNNDeep":
        model = Cifar10CNNDeep(num_classes=cfg["model"]["num_classes"]).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg["optimizer"]["lr"],
        weight_decay=cfg["optimizer"]["weight_decay"],
    )

    models_dir, logs_dir = ensure_outputs(cfg["paths"])
    best_accuracy = 0.0
    log_path = logs_dir / "train_log.csv"

    with open(log_path, "w", newline="", encoding="utf-8") as log_file:
        writer = csv.writer(log_file)
        writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])

        for epoch in range(1, cfg["training"]["epochs"] + 1):
            train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_metrics = evaluate(model, test_loader, device, criterion)

            writer.writerow(
                [
                    epoch,
                    f"{train_metrics['loss']:.4f}",
                    f"{train_metrics['accuracy']:.2f}",
                    f"{val_metrics['loss']:.4f}",
                    f"{val_metrics['accuracy']:.2f}",
                ]
            )

            print(
                f"Epoch {epoch}/{cfg['training']['epochs']} | "
                f"Train Loss: {train_metrics['loss']:.4f} | "
                f"Train Acc: {train_metrics['accuracy']:.2f}% | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.2f}%"
            )

            if val_metrics["accuracy"] > best_accuracy:
                best_accuracy = val_metrics["accuracy"]
                torch.save(model.state_dict(), models_dir / "best_model.pth")

    torch.save(model.state_dict(), models_dir / "last_model.pth")
    print(f"Best validation accuracy: {best_accuracy:.2f}%")


if __name__ == "__main__":
    main()

Overwriting src/train/train.py


## Train Model with Label Smoothing

### Subtask:
Train the Cifar10CNN model using the modified `src/train/train.py` and the existing `configs/config.yaml` to observe the effect of Label Smoothing on training and validation accuracy.


**Reasoning**:
Now that the loss function in `src/train/train.py` has been updated with label smoothing, I will train the model using the provided configuration to observe the impact on training and validation accuracy.



In [ ]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.8219 | Train Acc: 37.43% | Val Loss: 1.6604 | Val Acc: 46.59%
Epoch 2/20 | Train Loss: 1.6474 | Train Acc: 47.35% | Val Loss: 1.5994 | Val Acc: 50.62%
Epoch 3/20 | Train Loss: 1.5870 | Train Acc: 50.72% | Val Loss: 1.5046 | Val Acc: 55.12%
Epoch 4/20 | Train Loss: 1.5421 | Train Acc: 53.42% | Val Loss: 1.5039 | Val Acc: 55.68%
Epoch 5/20 | Train Loss: 1.5085 | Train Acc: 54.83% | Val Loss: 1.4399 | Val Acc: 58.65%
Epoch 6/20 | Train Loss: 1.4844 | Train Acc: 56.24% | Val Loss: 1.4609 | Val Acc: 58.19%
Epoch 7/20 | Train Loss: 1.4619 | Train Acc: 57.40% | Val Loss: 1.4660 | Val Acc: 58.26%
Epoch 8/20 | Train Loss: 1.4418 | Train Acc: 58.70% | Val Loss: 1.4571 | Val Acc: 58.41%
Epoch 9/20 | Train Loss: 1.4250 | Train Acc: 59.37% | Val Loss: 1.5005 | Val Acc: 57.15%
Epoch 10/20 | Train Loss: 1.4095 | Train Acc: 60.07% | Val Loss: 1.3516 | Val Acc: 63.16%
Epoch 11/20 | Train Loss: 1.3939 | Train Acc: 61.02% | Val Loss: 1.3817 | Val Acc: 62.70%
Epoch 12/20 | Train

### Summary of Changes and Training Results

**1. Changes Made to `src/train/train.py` (Loss Function):**
- The loss function in `src/train/train.py` was modified from a standard `nn.CrossEntropyLoss()` to `nn.CrossEntropyLoss(label_smoothing=0.1)`. This introduces label smoothing during training, which can help prevent the model from becoming overconfident and improve generalization.

**2. Training Results with Label Smoothing:**
After training the `Cifar10CNN` model (which was selected in the `configs/config.yaml` file) for 20 epochs with the updated data transformations and the new label smoothing loss function, the model achieved a **best validation accuracy of 66.48%**.

**Insights:**
- The introduction of label smoothing aims to regularize the model and prevent overfitting by softening the target labels. Comparing this result with previous runs of the `Cifar10CNN` model (e.g., 67.61% without label smoothing), it appears that label smoothing, in this specific configuration, slightly reduced the peak validation accuracy. However, label smoothing can often lead to better generalization in the long run and might prevent extreme overfitting in more complex scenarios or with more training epochs. Further hyperparameter tuning (e.g., the `label_smoothing` value, learning rate, or number of epochs) could potentially yield better results.

## Summarize Changes and Results

### Subtask:
Provide a summary of the changes made to `src/train/train.py` (specifically the addition of Label Smoothing) and the training results, including the best validation accuracy achieved.


## Summary of Changes and Training Results

### Q&A
1.  **What changes were made to `src/train/train.py`?**
    The loss function instantiation in `src/train/train.py` was modified to include label smoothing. Specifically, `criterion = nn.CrossEntropyLoss()` was changed to `criterion = nn.CrossEntropyLoss(label_smoothing=0.1)`.

2.  **What were the training results after this change?**
    After training the `Cifar10CNN` model with the added data augmentations and label smoothing for 20 epochs, the best validation accuracy achieved was **66.48%**.

3.  **How did this change impact the model's performance?**
    The previous run with `Cifar10CNN` and new data augmentations (but without label smoothing) achieved a best validation accuracy of 67.61%. With label smoothing at 0.1, the accuracy slightly decreased to 66.48%. While label smoothing generally helps with generalization and robustness, in this specific instance with the current model and data augmentation, it appears to have had a minor negative impact on the peak validation accuracy. This might suggest that the model was already well-regularized or that the chosen smoothing factor was not optimal for this setup, or that the model's capacity was not enough to capture the additional information.

## Final Task

### Subtask:
Summarize the entire process, including the modification of train.py for label smoothing and the outcome of the training.


## Summary:

### Q&A
1.  **What modifications were made to `src/train/train.py`?**
    The loss function definition in `src/train/train.py` was modified from `criterion = nn.CrossEntropyLoss()` to `criterion = nn.CrossEntropyLoss(label_smoothing=0.1)`. This change introduced label smoothing to the training process.
2.  **What was the outcome of the training after these modifications?**
    After training the Cifar10CNN model for 20 epochs with label smoothing, the model achieved a best validation accuracy of **66.48%**.
3.  **How did label smoothing impact performance compared to previous runs?**
    The best validation accuracy of 66.48% with label smoothing represents a slight decrease compared to a previous run (without label smoothing but with data augmentations), which achieved **67.61%** validation accuracy.

### Data Analysis Key Findings
*   The `src/train/train.py` script was updated to incorporate label smoothing, specifically setting `label_smoothing=0.1` within the `nn.CrossEntropyLoss` function.
*   Upon training the Cifar10CNN model with this modification for 20 epochs, the best validation accuracy achieved was **66.48%**.
*   The training process showed an improvement from an initial validation accuracy of 46.59% (Epoch 1) to a final validation accuracy of 63.54% (Epoch 20).
*   Comparing this result to a prior training run of the Cifar10CNN model (which achieved 67.61% validation accuracy without label smoothing but with similar data augmentations), the introduction of label smoothing at a factor of 0.1 led to a marginal reduction in peak validation accuracy.

### Insights or Next Steps
*   While label smoothing is a known regularization technique, in this specific scenario with the current model and hyperparameter choice, it resulted in a slight reduction in peak validation accuracy. Further investigation into optimal `label_smoothing` values (e.g., trying 0.05 or 0.2) or assessing its impact on generalization and robustness over more training epochs or different datasets could be beneficial.
*   The current model configuration might already be sufficiently regularized, or the chosen `label_smoothing` factor was not optimal, suggesting that further hyperparameter tuning for label smoothing is warranted to determine if it can offer benefits to generalization without sacrificing peak accuracy.


In [2]:
!cat src/train/train.py
!cat configs/config.yaml

import argparse
import csv
from datetime import datetime
from pathlib import Path

import torch
import yaml
from torch import nn, optim

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import Cifar10CNN
from src.train.eval import evaluate
from src.utils.metrics import batch_accuracy


def load_config(config_path: str):
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += batch_accuracy(logits, labels)

    return {
        "loss": running_loss / len(loader),
        "accuracy": 10

In [17]:
%%writefile src/models/cnn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


class ResNetLite(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.initial_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        # The rest of the ResNetLite architecture will be added here later
        # Corrected input features for the fc layer: 64 * 16 * 16 = 16384
        self.fc = nn.Linear(64 * 16 * 16, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.initial_layers(x)
        # Placeholder for residual blocks
        x = x.view(x.size(0), -1) # Flatten for now
        return self.fc(x)

Overwriting src/models/cnn.py


**Reasoning**:
Now that the `ResNetLite` model definition in `src/models/cnn.py` has been corrected and the `configs/config.yaml` is set to use `ResNetLite`, I will re-train the model to ensure the `RuntimeError` is resolved and to evaluate its performance.

In [18]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 2.3569 | Train Acc: 34.24% | Val Loss: 1.8192 | Val Acc: 43.97%
Epoch 2/20 | Train Loss: 1.8183 | Train Acc: 41.71% | Val Loss: 1.5820 | Val Acc: 50.06%
Epoch 3/20 | Train Loss: 1.6701 | Train Acc: 46.67% | Val Loss: 1.4179 | Val Acc: 55.90%
Epoch 4/20 | Train Loss: 1.5937 | Train Acc: 48.89% | Val Loss: 1.4312 | Val Acc: 55.16%
Epoch 5/20 | Train Loss: 1.5246 | Train Acc: 51.33% | Val Loss: 1.3495 | Val Acc: 58.24%
Epoch 6/20 | Train Loss: 1.4333 | Train Acc: 54.87% | Val Loss: 1.2637 | Val Acc: 62.23%
Epoch 7/20 | Train Loss: 1.4080 | Train Acc: 55.67% | Val Loss: 1.2833 | Val Acc: 61.58%
Epoch 8/20 | Train Loss: 1.3998 | Train Acc: 56.21% | Val Loss: 1.2652 | Val Acc: 62.14%
Epoch 9/20 | Train Loss: 1.4044 | Train Acc: 56.12% | Val Loss: 1.2486 | Val Acc: 62.78%
Epoch 10/20 | Train Loss: 1.3880 | Train Acc: 56.70% | Val Loss: 1.3022 | Val Acc: 60.84%
Epoch 11/20 | Train Loss: 1.3396 | Train Acc: 58.63% | Val Loss: 1.2387 | Val Acc: 63.32%
Epoch 12/20 | Train

In [4]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update epochs
config['training']['epochs'] = 20

# Update mean and std values in the dataset section
config['dataset']['mean'] = [0.4914, 0.4822, 0.4465]
config['dataset']['std'] = [0.247, 0.243, 0.261]

# Ensure model name is Cifar10CNN as per train.py
config['model']['name'] = 'Cifar10CNN'

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} with new settings: epochs={config['training']['epochs']}, mean={config['dataset']['mean']}, std={config['dataset']['std']}, model={config['model']['name']}.")

Updated configs/config.yaml with new settings: epochs=20, mean=[0.4914, 0.4822, 0.4465], std=[0.247, 0.243, 0.261], model=Cifar10CNN.


**Reasoning**:
Now that `src/train/train.py` has been updated to include `ResNetLite` in the model selection and `configs/config.yaml` specifies `ResNetLite`, I will train the model using the updated script and configuration.

In [16]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/project_CNN_CIFAR10/src/train/train.py", line 130, in <module>
    main()
  File "/content/project_CNN_CIFAR10/src/train/train.py", line 97, in main
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/project_CNN_CIFAR10/src/train/train.py", line 31, in train_one_epoch
    logits = model(images)
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

**Reasoning**:
Before modifying `src/train/train.py` to include `ResNetLite` in the model selection logic, I need to display its current content to understand the existing structure and correctly place the new import and conditional logic.

In [14]:
with open('src/train/train.py', 'r') as f:
    print(f.read())

import argparse
import csv
from pathlib import Path

import torch
import yaml
from torch import nn, optim
from torch.optim import lr_scheduler # Import lr_scheduler

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import Cifar10CNN
from src.train.eval import evaluate
from src.utils.metrics import batch_accuracy


def load_config(config_path: str):
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += batch_accuracy(logits, labels)

    return {
        "loss": running_loss / len(lo

**Reasoning**:
Now that the `ResNetLite` class is defined, I need to update the `configs/config.yaml` file to use this new model. This involves changing the `model_name` parameter to `ResNetLite`.

In [13]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update the model name to use the new ResNetLite model
config['model']['name'] = 'ResNetLite'

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print(f"Updated {config_path} to use model: {config['model']['name']}.")

Updated configs/config.yaml to use model: ResNetLite.


**Reasoning**:
Now that the `src/models/cnn.py` has been overwritten with the `ResNetLite` class, I will display the file content to ensure the changes were applied correctly.

In [12]:
!cat src/models/cnn.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)


clas

**Reasoning**:
To add the initial convolutional and pooling layers for the `ResNetLite` architecture, I will read the content of `src/models/cnn.py` to ensure I have the latest version of the file, then create a new code block that overwrites the file content with the updated `src/models/cnn.py` which includes the new `ResNetLite` class.

In [10]:
with open('src/models/cnn.py', 'r') as f:
    print(f.read())

import torch
import torch.nn as nn
import torch.nn.functional as F


class Cifar10CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)



In [5]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

100% 170M/170M [00:13<00:00, 12.8MB/s]
Epoch 1/20 | Train Loss: 1.7618 | Train Acc: 37.52% | Val Loss: 1.5640 | Val Acc: 46.64%
Epoch 2/20 | Train Loss: 1.5514 | Train Acc: 47.55% | Val Loss: 1.8533 | Val Acc: 39.58%
Epoch 3/20 | Train Loss: 1.4826 | Train Acc: 50.99% | Val Loss: 1.3826 | Val Acc: 55.75%
Epoch 4/20 | Train Loss: 1.4321 | Train Acc: 53.14% | Val Loss: 1.3668 | Val Acc: 55.54%
Epoch 5/20 | Train Loss: 1.3968 | Train Acc: 54.82% | Val Loss: 1.3716 | Val Acc: 55.21%
Epoch 6/20 | Train Loss: 1.3697 | Train Acc: 56.28% | Val Loss: 1.4213 | Val Acc: 55.69%
Epoch 7/20 | Train Loss: 1.3421 | Train Acc: 57.57% | Val Loss: 1.3186 | Val Acc: 57.29%
Epoch 8/20 | Train Loss: 1.3253 | Train Acc: 57.95% | Val Loss: 1.2710 | Val Acc: 60.62%
Epoch 9/20 | Train Loss: 1.3024 | Train Acc: 59.44% | Val Loss: 1.2263 | Val Acc: 63.03%
Epoch 10/20 | Train Loss: 1.2870 | Train Acc: 59.88% | Val Loss: 1.2120 | Val Acc: 63.69%
Epoch 11/20 | Train Loss: 1.2714 | Train Acc: 60.78% | Val Loss: 1.408

In [8]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.7738 | Train Acc: 36.69% | Val Loss: 1.5936 | Val Acc: 44.67%
Epoch 2/20 | Train Loss: 1.5665 | Train Acc: 46.79% | Val Loss: 1.4670 | Val Acc: 51.22%
Epoch 3/20 | Train Loss: 1.4901 | Train Acc: 50.64% | Val Loss: 1.4600 | Val Acc: 51.52%
Epoch 4/20 | Train Loss: 1.4343 | Train Acc: 53.14% | Val Loss: 1.5035 | Val Acc: 51.41%
Epoch 5/20 | Train Loss: 1.4003 | Train Acc: 54.64% | Val Loss: 1.5241 | Val Acc: 51.11%
Epoch 6/20 | Train Loss: 1.3495 | Train Acc: 57.52% | Val Loss: 1.3990 | Val Acc: 54.64%
Epoch 7/20 | Train Loss: 1.3359 | Train Acc: 57.92% | Val Loss: 1.3315 | Val Acc: 58.40%
Epoch 8/20 | Train Loss: 1.3174 | Train Acc: 58.80% | Val Loss: 1.2435 | Val Acc: 61.51%
Epoch 9/20 | Train Loss: 1.3087 | Train Acc: 58.90% | Val Loss: 1.2641 | Val Acc: 60.80%
Epoch 10/20 | Train Loss: 1.2984 | Train Acc: 59.47% | Val Loss: 1.3423 | Val Acc: 57.73%
Epoch 11/20 | Train Loss: 1.2706 | Train Acc: 60.69% | Val Loss: 1.2737 | Val Acc: 60.27%
Epoch 12/20 | Train

In [6]:
with open('src/train/train.py', 'r') as f:
    print(f.read())

import argparse
import csv
from pathlib import Path

import torch
import yaml
from torch import nn, optim

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import Cifar10CNN
from src.train.eval import evaluate
from src.utils.metrics import batch_accuracy


def load_config(config_path: str):
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += batch_accuracy(logits, labels)

    return {
        "loss": running_loss / len(loader),
        "accuracy": 100.0 * running_acc / len(loader

In [9]:
import os

# Ensure the current working directory is the project root
if os.getcwd() != '/content/project_CNN_CIFAR10':
    os.chdir('/content/project_CNN_CIFAR10')

!python -m src.train.train --config configs/config.yaml

Epoch 1/20 | Train Loss: 1.7596 | Train Acc: 37.37% | Val Loss: 1.7324 | Val Acc: 40.25%
Epoch 2/20 | Train Loss: 1.5610 | Train Acc: 47.22% | Val Loss: 1.4601 | Val Acc: 51.33%
Epoch 3/20 | Train Loss: 1.4822 | Train Acc: 50.93% | Val Loss: 1.5208 | Val Acc: 50.85%
Epoch 4/20 | Train Loss: 1.4323 | Train Acc: 53.39% | Val Loss: 1.3610 | Val Acc: 55.99%
Epoch 5/20 | Train Loss: 1.3943 | Train Acc: 55.01% | Val Loss: 1.4982 | Val Acc: 52.26%
Epoch 6/20 | Train Loss: 1.3433 | Train Acc: 57.30% | Val Loss: 1.3634 | Val Acc: 56.86%
Epoch 7/20 | Train Loss: 1.3228 | Train Acc: 58.16% | Val Loss: 1.3415 | Val Acc: 57.96%
Epoch 8/20 | Train Loss: 1.3081 | Train Acc: 58.93% | Val Loss: 1.2934 | Val Acc: 60.20%
Epoch 9/20 | Train Loss: 1.3020 | Train Acc: 59.42% | Val Loss: 1.2355 | Val Acc: 61.68%
Epoch 10/20 | Train Loss: 1.2854 | Train Acc: 60.01% | Val Loss: 1.2811 | Val Acc: 60.98%
Epoch 11/20 | Train Loss: 1.2609 | Train Acc: 61.12% | Val Loss: 1.2146 | Val Acc: 63.26%
Epoch 12/20 | Train